## Problem

Floating-point arithmetic is quietly catastrophic in financial software.

```python
>>> 0.1 + 0.2
0.30000000000000004
>>> round(9.995, 2)
9.99  # ← should be 10.00
```

Your task is to implement a `MoneyAmount` dataclass that stores monetary values with **decimal precision** and supports **two distinct rounding strategies** — rounding and truncation — plus **currency-safe addition**.

---

## Requirements

Implement the `MoneyAmount` class with the following interface:

### Constructor & Factory

| Method                                                | Description                                                                |
| ----------------------------------------------------- | -------------------------------------------------------------------------- |
| `MoneyAmount(amount: Decimal, currency: str)`         | Direct construction from a `Decimal`                                       |
| `MoneyAmount.from_float(value: float, currency: str)` | Safe construction from a `float` — must not inherit float precision errors |

### Rounding methods

| Method                | Behaviour                                                                                |
| --------------------- | ---------------------------------------------------------------------------------------- |
| `.round_to_cent()`    | Round to 2 decimal places using **half-up** (banker's rounding is _not_ acceptable here) |
| `.truncate_to_cent()` | Truncate toward zero — never round up                                                    |

Both methods must return a **new `MoneyAmount`** (immutable pattern).

### Arithmetic

| Operation | Behaviour                                                             |
| --------- | --------------------------------------------------------------------- |
| `a + b`   | Add two `MoneyAmount` values; raise `ValueError` if currencies differ |

## Examples

```python
price = MoneyAmount(Decimal("9.995"), "USD")

price.round_to_cent()
# MoneyAmount(amount=Decimal('10.00'), currency='USD')

price.truncate_to_cent()
# MoneyAmount(amount=Decimal('9.99'), currency='USD')

MoneyAmount(Decimal("4.50"), "USD") + MoneyAmount(Decimal("5.50"), "USD")
# MoneyAmount(amount=Decimal('10.00'), currency='USD')

MoneyAmount(Decimal("10.00"), "USD") + MoneyAmount(Decimal("5.00"), "EUR")
# ValueError: Cannot add USD and EUR
```

## Constraints

- Use **only the Python standard library** (`decimal`, `dataclasses`).
- `from_float` must route through `str` conversion — direct `Decimal(float)` is forbidden.
- `round_to_cent` must use `ROUND_HALF_UP` — `9.995` must become `10.00`, not `9.99`.
- `truncate_to_cent` must use `ROUND_DOWN` — `9.999` must become `9.99`, not `10.00`.
- `MoneyAmount` must be a `@dataclass`.

## Hints
- Hint 1 — Why not `round()`?
    - Python's built-in `round()` uses **banker's rounding** (round half to even). That means `round(2.5)` → `2`, not `3`. For financial contexts you need `ROUND_HALF_UP` from the `decimal` module.
- Hint 2 — Why `Decimal(str(value))` in `from_float`?
    - `Decimal(9.995)` gives you `Decimal('9.99499999999999955591...')` because that's how the float is stored in IEEE 754. `Decimal(str(9.995))` gives you `Decimal('9.995')` — the number you actually typed.
- Hint 3 — What is `quantize`?
    - `Decimal.quantize(exp, rounding)` forces a `Decimal` to the precision of `exp`. Use `Decimal("0.01")` as the target exponent to get exactly 2 decimal places.


## 80/20 Insight

The `decimal` module's `quantize` + rounding constants pattern covers **the vast majority of real-world financial rounding bugs** in Python. Master this one idiom and you've eliminated the most common class of money-handling errors you'll encounter in production.

## Solution

In [ ]:
from decimal import Decimal, ROUND_HALF_UP, ROUND_DOWN
from dataclasses import dataclass

CENT = Decimal('0.01')

In [ ]:
@dataclass
class MoneyAmount:
    amount: Decimal
    currency: str

    @classmethod
    def from_float(cls, value:float, currency:str) -> 'MoneyAmount':
        # Always got through string yo avoid float representation errors
        return cls(Decimal(str(value)), currency)
    
    def round_to_cent(self) -> 'MoneyAmount':
        return MoneyAmount(
            self.amount.quantize(CENT, rounding=ROUND_HALF_UP),
            self.currency
        )
    
    def truncate_to_cent(self) -> 'MoneyAmount':
        return MoneyAmount(
            self.amount.quantize(CENT, rounding=ROUND_DOWN),
            self.currency
        )
    
    def __add__(self, other: 'MoneyAmount') -> 'MoneyAmount':
        if self.currency != other.currency:
            raise ValueError(f"Cannot add {self.currency} to {other.currency}")
        return MoneyAmount(self.amount + other.amount, self.currency)

In [ ]:
price = MoneyAmount.from_float(9.995, 'USD')
print(price.round_to_cent())  # Should print 10.00 USD
print(price.truncate_to_cent())  # Should print 9.99 USD

In [ ]:
from decimal import Decimal
import unittest

class TestMoneyAmount(unittest.TestCase):

    def test_round_half_up(self):
        m = MoneyAmount(Decimal("9.995"), "USD")
        self.assertEqual(m.round_to_cent().amount, Decimal("10.00"))

    def test_truncate_does_not_round_up(self):
        m = MoneyAmount(Decimal("9.999"), "USD")
        self.assertEqual(m.truncate_to_cent().amount, Decimal("9.99"))

    def test_from_float_avoids_ieee754_error(self):
        m = MoneyAmount.from_float(9.995, "USD")
        self.assertEqual(m.round_to_cent().amount, Decimal("10.00"))

    def test_addition_same_currency(self):
        a = MoneyAmount(Decimal("4.50"), "USD")
        b = MoneyAmount(Decimal("5.50"), "USD")
        result = a + b
        self.assertEqual(result.amount, Decimal("10.00"))
        self.assertEqual(result.currency, "USD")

    def test_addition_currency_mismatch(self):
        a = MoneyAmount(Decimal("10.00"), "USD")
        b = MoneyAmount(Decimal("5.00"), "EUR")
        with self.assertRaisesRegex(ValueError, "Cannot add USD to EUR"):
            _ = a + b

    def test_immutability_of_rounding(self):
        original = MoneyAmount(Decimal("9.995"), "USD")
        rounded = original.round_to_cent()
        self.assertEqual(original.amount, Decimal("9.995"))
        self.assertEqual(rounded.amount, Decimal("10.00"))

unittest.main(argv=[""], verbosity=2, exit=False)
